# 01 — Download & Verify Repositories

**Purpose.** Clone the two real repositories this project builds on, verify
their structure against paths confirmed by direct inspection (not assumed),
and — most importantly — empirically confirm the checkpoint-availability
finding before any later notebook relies on it.

- Dheur & Ben Taieb, AISTATS 2024 -- `quantile-recalibration-training`
- Dheur & Ben Taieb, ICML 2023 (base study) -- `probabilistic-calibration-study`

**Expected runtime:** under 1 minute (two shallow clones + file checks).
**GPU:** not used in this notebook.

Cloned into `external/` (already in `.gitignore` -- these are third-party
repos, not vendored into this project's own git history, and are **not**
synced via Google Drive the way your own project files are — they're
fetched fresh from GitHub every time this notebook runs).


## Step 1 — Locate project & import helpers

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError(
            "PROJECT_ROOT not set and could not be located. Run 00_environment.ipynb first."
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import env_utils

EXTERNAL_DIR = PROJECT_ROOT / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"EXTERNAL_DIR = {EXTERNAL_DIR}")


## Step 2 — Clone both repositories

In [ ]:
REPOS = {
    "quantile-recalibration-training": "https://github.com/Vekteur/quantile-recalibration-training.git",
    "probabilistic-calibration-study": "https://github.com/Vekteur/probabilistic-calibration-study.git",
}

REPO_PATHS = {}
for name, url in REPOS.items():
    dest = EXTERNAL_DIR / name
    ok = env_utils.clone_or_pull_repo(repo_url=url, dest=dest, branch="main")
    REPO_PATHS[name] = dest
    print(f"{name}: {'OK' if ok or (dest / '.git').exists() else 'FAILED'} -> {dest}")

QRT_DIR = REPO_PATHS["quantile-recalibration-training"]
PCS_DIR = REPO_PATHS["probabilistic-calibration-study"]


## Step 3 — Verify structure against paths confirmed by direct inspection

These exact paths were confirmed by cloning both repos and reading their
contents directly (not guessed from the paper text alone) — e.g. the paper
text alone would not have revealed that `demo/datamodule.py` uses a
**different** train/val/calib/test split than the actual large-scale study
in `uq/`.

In [ ]:
from dataclasses import dataclass

@dataclass
class CheckResult:
    path: str
    exists: bool
    critical: bool

def check_paths(base: Path, specs: list[tuple[str, bool]]) -> list[CheckResult]:
    return [CheckResult(rel, (base / rel).exists(), critical) for rel, critical in specs]

qrt_checks = check_paths(QRT_DIR, [
    ("run.py", True),
    ("requirements.txt", True),
    ("demo/QRT.py", True),
    ("demo/datamodule.py", True),
    ("uq/configs/dataset_groups.py", True),
    ("uq/models/general/mlp.py", True),
    ("uq/models/pred_type/mixture_dist.py", True),
    ("uq/datamodules/base_datamodule.py", True),
    ("uq/utils/checkpoints.py", True),
    ("README.md", False),
])

pcs_checks = check_paths(PCS_DIR, [
    ("requirements.txt", True),
    ("README.md", False),
])

print("=== quantile-recalibration-training ===")
qrt_ok = True
for c in qrt_checks:
    status = "OK" if c.exists else ("MISSING (critical)" if c.critical else "missing (non-critical)")
    print(f"  [{status:22s}] {c.path}")
    if c.critical and not c.exists:
        qrt_ok = False

print("\n=== probabilistic-calibration-study ===")
pcs_ok = True
for c in pcs_checks:
    status = "OK" if c.exists else ("MISSING (critical)" if c.critical else "missing (non-critical)")
    print(f"  [{status:22s}] {c.path}")
    if c.critical and not c.exists:
        pcs_ok = False

if not (qrt_ok and pcs_ok):
    raise RuntimeError(
        "One or more critical files are missing. The upstream repos may have "
        "changed structure since this notebook was written -- inspect manually "
        "before continuing to notebook 02."
    )
print("\nAll critical paths verified.")


## Step 4 — Confirm the checkpoint-availability finding empirically

Don't just take a prior read-through's word for it — actually search the
cloned repos for any shipped checkpoint files.

In [ ]:
ckpt_extensions = (".ckpt", ".pt", ".pth")
found_checkpoints = []
for repo_dir in (QRT_DIR, PCS_DIR):
    for ext in ckpt_extensions:
        found_checkpoints.extend(repo_dir.rglob(f"*{ext}"))

if found_checkpoints:
    print(f"[!] Found {len(found_checkpoints)} checkpoint-like file(s) -- inspect these, "
          "the 'no pretrained checkpoints shipped' assumption may need revisiting:")
    for f in found_checkpoints:
        print(f"    {f.relative_to(EXTERNAL_DIR)}")
else:
    print("Confirmed: no .ckpt/.pt/.pth files anywhere in either cloned repo.")
    print("-> Notebook 03 must TRAIN base models itself; it cannot download pretrained weights.")
    print("-> Their own README documents `remove_checkpoints=True` in the main experiment "
          "command, consistent with this.")


## Step 5 — Real pilot dataset group & training entrypoint

Confirmed from `uq/configs/dataset_groups.py`: the `uci` group has 12 small
datasets and uses `UCIDataModule`, which downloads directly from UCI URLs
-- no OpenML account/API key needed, unlike the other three dataset groups
(`oml_297`, `oml_299`, `oml_269`). This makes it the natural first pilot.

In [ ]:
import ast

dg_path = QRT_DIR / "uq" / "configs" / "dataset_groups.py"
source = dg_path.read_text()

# Extract the `names = [...]` list inside `uci_config` without executing the
# file (it imports the full uq package, which we haven't installed a config
# context for yet at this stage) -- static parse instead.
tree = ast.parse(source)
uci_names = None
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef) and node.name == "uci_config":
        for sub in ast.walk(node):
            if isinstance(sub, ast.Assign) and any(
                isinstance(t, ast.Name) and t.id == "names" for t in sub.targets
            ):
                uci_names = ast.literal_eval(sub.value)

print(f"uci group datasets ({len(uci_names)}): {uci_names}")

expected = ["CPU", "Yacht", "MPG", "Energy", "Crime", "Fish",
            "Concrete", "Airfoil", "Kin8nm", "Power", "Naval", "Protein"]
assert uci_names == expected, (
    f"uci group in the cloned repo differs from what config.yaml expects.\n"
    f"  repo:     {uci_names}\n  expected: {expected}\n"
    "The upstream repo may have changed -- update configs/config.yaml before continuing."
)
print("\nMatches configs/config.yaml -- consistent.")

print(f'''
Real training entrypoint (from {QRT_DIR / "run.py"}), a light pilot invocation
for notebook 03 will be based on:

  python run.py name="pilot" nb_workers=1 repeat_tuning=1 \\
      log_base_dir="{{PROJECT_ROOT}}/checkpoints" progress_bar=False \\
      save_train_metrics=False save_val_metrics=False remove_checkpoints=False \\
      selected_dataset_groups=["uci"] tuning_type="QRT"

Differences from their full-scale command: repeat_tuning=1 (not 5, for pilot
speed) and remove_checkpoints=False (we need the checkpoints -- that's the
whole point).
''')


## Summary & next steps

Verified this session:
- Both repos clone successfully and match the expected (directly-inspected)
  structure.
- No pretrained checkpoints ship with either repo — confirmed by an actual
  filesystem search, not assumption.
- The `uci` dataset group (12 datasets) is confirmed correct against
  `configs/config.yaml` and is the recommended pilot.
- The real training entrypoint and a pilot-appropriate invocation are
  printed above for `02`/`03` to use.

**Next:** `02_prepare_datasets.ipynb` — materializes the `uci` group's 12
datasets locally using the real `UCIDataModule` and the verified
65/10/15/10 split, independent of the heavier Hydra `run.py` orchestration
where possible, so it stays inspectable in notebook cells rather than a
single opaque CLI call.
